<a href="https://colab.research.google.com/github/lawrence-kagugo/kenya-telecom-churn-analysis/blob/main/notebooks/01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kenya Telecom Customer Churn - Data Understanding
**Project:** End-to-End Customer Churn Intelligence Analysis

**Author:** Lawrence Kagugo
**Step:** 5 - Data Understanding

In [18]:
# Import pandas — the core library for loading and exploring tabular data.
# We alias it as 'pd' by universal convention, so every pandas function is called as pd.something()
import pandas as pd

# Load the dataset directly from our GitHub repo's raw file URL.
# Loading from GitHub (rather than uploading manually each time) makes the notebook
# reproducible — anyone who clones the repo can re-run this cell and get the same data,
# without needing to manually upload a file first.
url = "https://raw.githubusercontent.com/lawrence-kagugo/kenya-telecom-churn-analysis/refs/heads/main/data/raw/kenya_telecom_customer_churn_dataset.csv"
df = pd.read_csv(url)

# Quick confirmation the load worked — shape tells us (rows, columns)
print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (5000, 70)


In [19]:
# .info() gives us three things at once: column names, non-null counts, and data types.
# This is usually the very first command run on any new dataset — it tells us
# immediately whether pandas interpreted each column correctly (e.g. dates as
# strings vs actual dates) and flags any columns with missing values.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 70 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CustomerID                 5000 non-null   object 
 1   FirstName                  5000 non-null   object 
 2   LastName                   5000 non-null   object 
 3   Gender                     5000 non-null   object 
 4   Age                        5000 non-null   int64  
 5   DateOfBirth                5000 non-null   object 
 6   MaritalStatus              5000 non-null   object 
 7   NumberOfDependents         5000 non-null   int64  
 8   Occupation                 4915 non-null   object 
 9   EducationLevel             4917 non-null   object 
 10  Country                    5000 non-null   object 
 11  County                     5000 non-null   object 
 12  Town                       5000 non-null   object 
 13  PostalCode                 5000 non-null   int64

## Data Understanding - Initial Findings

**Finding 1: Date columns loaded as text**
`DateOfBirth`, `JoinDate`, and `ChurnDate` are currently stored as `object` (text)
instead of `datetime`. This will need to be converted during Data Cleaning
so we can perform date-based calculations (e.g. tenure verification, churn timing analysis).

In [20]:
# .value_counts() counts how many times each unique value appears in a column.
# For Churn (Yes/No), this tells us exactly how many customers churned vs stayed.
print(df['Churn'].value_counts())

# normalize=True converts the counts into proportions (percentages) instead of raw counts.
# This is what we actually quote in reports — "21% churn rate" is more useful
# to a stakeholder than "1072 customers churned" on its own, since it's comparable
# across time periods or company sizes.
print()
print(df['Churn'].value_counts(normalize=True) * 100)

Churn
No     3928
Yes    1072
Name: count, dtype: int64

Churn
No     78.56
Yes    21.44
Name: proportion, dtype: float64


**Finding 2: Churn rate**
Overall churn rate is 21.44% (1,072 of 5,000 customers). This is a moderately
imbalanced split - worth noting for any future statistical testing or modeling,
since raw accuracy would be a misleading metric on its own.

In [21]:
# .groupby('Churn') splits the dataframe into two groups: churned and retained.
# .mean() then calculates the average of each score column within each group.
# This lets us directly compare: is the average score meaningfully different
# between the two groups, in the direction we hypothesized?
df.groupby('Churn')[['ChurnRiskScore', 'CustomerHealthScore', 'EngagementScore']].mean()

,ChurnRiskScore,CustomerHealthScore,EngagementScore
Churn,,,
No,39.712424,76.613875,53.194883
Yes,62.410634,61.274720,38.308769


**Finding 3: Risk/health/engagement scores validated**
Average ChurnRiskScore is notably higher for churned customers (62.4 vs 39.7),
while CustomerHealthScore (61.3 vs 76.6) and EngagementScore (38.3 vs 53.2) are
both notably lower. This confirms these pre-built scores behave as expected and
can be used with confidence for risk segmentation later in the project.

# Sum CustomerLifetimeValue separately for churned vs retained customers.
# This tells us the total dollar (KES) value tied up in each group -
# specifically, how much value has already walked out the door due to churn.

In [22]:
df.groupby('Churn')['CustomerLifetimeValue'].sum()

,CustomerLifetimeValue
Churn,
No,4.676517e+08
Yes,1.329767e+08


In [23]:
churned_clv = 132976700  # or however precise you want
retained_clv = 467651700
pct_at_risk = (churned_clv / (churned_clv + retained_clv)) * 100
print(pct_at_risk)

22.139595796668956


**Finding 4: Revenue at risk**
Churned customers account for KES 132.98M in cumulative CustomerLifetimeValue,
against KES 467.65M for retained customers - approximately 22.14% of total CLV
lost to churn. This closely tracks the 21.44% churn rate, suggesting churned
customers are not disproportionately high- or low-value on average (worth
checking further at the segment level in EDA).

In [24]:
# .unique() lists every distinct value in a column.
# We're checking key categorical columns tied to our business questions
# (contract type, plan, segment, region) for typos, inconsistent casing,
# or unexpected categories before we trust any groupby analysis on them.
categorical_cols = ['ContractType', 'SubscriptionPlan', 'CustomerSegment', 'Region', 'SatisfactionCategory']

for col in categorical_cols:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

--- ContractType ---
['Month-to-Month' '24 Months' '12 Months']

--- SubscriptionPlan ---
['Standard' 'Premium' 'Family' 'Unlimited' 'Business' 'Basic' 'Enterprise']

--- CustomerSegment ---
['SME' 'Residential' 'Corporate' 'Government' 'Student']

--- Region ---
['Nyanza Region' 'Nairobi Region' 'Eastern Region' 'Western Region'
 'Rift Valley Region' 'Central Region' 'Coast Region'
 'North Eastern Region']

--- SatisfactionCategory ---
['Satisfied' 'Neutral' 'Very Satisfied' 'Unsatisfied']



**Finding 5: Categorical fields clean**
ContractType, SubscriptionPlan, CustomerSegment, Region, and SatisfactionCategory
were checked for typos, inconsistent casing, and duplicate-meaning categories.
All five are clean and ready for groupby analysis without correction.

In [25]:
# Calculate missing value percentage for every column, sorted highest to lowest.
# This gives us a clean, prioritized view instead of scanning the full .info() output.
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]  # only show columns that actually have missing data
print(missing_pct)

ChurnReason               78.56
ChurnDate                 78.56
ProductRating              1.84
Occupation                 1.70
EducationLevel             1.66
CommunityParticipation     1.54
AveragePurchaseValue       1.50
ProfitMarginPercent        1.50
ReferralSource             1.50
MarketingClickRate         1.48
PreferredCommunication     1.34
SurveyScore                1.30
AverageSessionMinutes      1.28
EmailOpenRate              1.24
AverageResolutionHours     1.18
dtype: float64


**Finding 6: Missing data pattern**
ChurnDate and ChurnReason (78.56% missing) reflect structural missingness -
expected, since these fields only apply to churned customers, not a data quality
issue. The remaining columns (all under 2% missing) show no obvious logical
explanation and will require individual imputation decisions during Data Cleaning.